
# LLM-judge STR-CoT synth answers with a vLLM-served Qwen3-4B model

Load the synth CoT samples, group them by `answer_check` outcome (correct vs. the two failure modes), and ask the served model to give a binary same/not-same score on the `low_overlap` group by comparing the ground-truth answer with the generated answer.

**Prerequisites:** launch the server with `bash bash_scripts/vllm_serving/qwen3_4b.sh`.

## Section 1: Load the synth CoT samples

In [2]:
import os
import glob
import json

STR_COT_SAMPLES_ROOT = "/scratch/rajendrakumar.a/visual-grounded-vlm/annotations/openo3video/openo3_str_cot_synth/samples"

# load every sample json into a list
samples = []
for path in sorted(glob.glob(os.path.join(STR_COT_SAMPLES_ROOT, "*.json"))):
    with open(path) as f:
        samples.append(json.load(f))
print(f"loaded {len(samples)} samples")

loaded 6758 samples


## Section 2: Group by answer_check outcome

`answer_check.reason` is one of `ok` (correct), `low_overlap`, or `no_answer_tag` (the two failure modes).

In [3]:
# bucket samples by their answer_check reason
correct = [s for s in samples if s["answer_check"]["reason"] == "ok"]
low_overlap = [s for s in samples if s["answer_check"]["reason"] == "low_overlap"]
no_answer_tag = [s for s in samples if s["answer_check"]["reason"] == "no_answer_tag"]

print(f"correct       : {len(correct)}")
print(f"low_overlap   : {len(low_overlap)}")
print(f"no_answer_tag : {len(no_answer_tag)}")

correct       : 6510
low_overlap   : 206
no_answer_tag : 42


## Section 3: LLM-judge the low_overlap group

For each `low_overlap` sample, pass the ground-truth answer and the generated answer to the model and ask for a binary score: do they convey the same answer? The judge is told to score on correctness, not length.

In [4]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
model = "Qwen/Qwen3-4B-Instruct-2507"

client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)

SYSTEM_PROMPT = (
    "You are a strict evaluator judging whether a generated answer matches the ground-truth "
    "answer to a question about a video.\n"
    "Judge ONLY on correctness of the content, not on length, verbosity, or phrasing: a short "
    "answer that is correct should score the same as a long one that conveys the same information.\n"
    "Output exactly 1 if the generated answer conveys the same correct answer as the ground "
    "truth, and exactly 0 otherwise. Output only the single digit, with no explanation."
)

In [5]:
def judge(question, gt_answer, gen_answer):
    """Ask the served model for a binary same/not-same score. Returns 1, 0, or None on parse failure."""
    user_prompt = (
        f"Question: {question}\n\n"
        f"Ground-truth answer: {gt_answer}\n\n"
        f"Generated answer: {gen_answer}\n\n"
        "Score (1 if same, 0 if not):"
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=4,
    )
    out = resp.choices[0].message.content.strip()
    return 1 if out.startswith("1") else 0 if out.startswith("0") else None

In [6]:
# score every low_overlap sample
results = []
for s in low_overlap:
    score = judge(s["question"], s["gt_answer"], s["extracted_answer"])
    results.append({"id": s["id"], "score": score})

n_same = sum(1 for r in results if r["score"] == 1)
n_diff = sum(1 for r in results if r["score"] == 0)
n_none = sum(1 for r in results if r["score"] is None)
print(f"judged {len(results)} low_overlap samples")
print(f"same (1)     : {n_same}")
print(f"not same (0) : {n_diff}")
print(f"unparsed     : {n_none}")

judged 206 low_overlap samples
same (1)     : 155
not same (0) : 51
unparsed     : 0


In [8]:
# display the not-same (score 0) samples alongside their answers
not_same = [r for r in results if r["score"] == 0]
print(f"{len(not_same)} not-same samples\n")
for r in not_same:
    s = next(x for x in low_overlap if x["id"] == r["id"])
    print(f"[{r['id']}] score={r['score']}")
    print(f"  Q  : {s['question']}")
    print(f"  GT : {s['gt_answer']}")
    print(f"  GEN: {s['extracted_answer']}")
    print()

51 not-same samples

[000036] score=0
  Q  : What color is the floating donut that the man is using?
  GT : The man is using a blue floating donut, while the woman is in a green one.
  GEN: green

[000560] score=0
  Q  : What is the name of the person credited in the video's closing sequence?
  GT : The person credited in the video's closing sequence is Felix Morrison.
  GEN: Felix Moris

[000634] score=0
  Q  : What kind of beverage is shown being prepared for the games?
  GT : Numerous pitchers filled with beer and topped with foam are shown lined up on a table, ready for the beer pong tournament.
  GEN: There is no visible beverage being prepared in the video.

[001406] score=0
  Q  : What is the man at the dinner table wearing on his head?
  GT : The man is wearing a red and white Santa hat while sitting at the dinner table with other people.
  GEN: The man is wearing a blue bandana on his head.

[001540] score=0
  Q  : What does the toy that the cat is playing with look like?
  GT